Pour les benchmark on va faire 3 approches :

1) Naïf Saisonnier (S-Naive) : La valeur prédite est celle de la même période l'année précédente (J-364).

2) Moyenne Mobile (Moving Average) : La prédiction est la moyenne des N derniers jours.


In [1]:
import os 
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_log_error

os.chdir("../")
os.getcwd()

'/Users/pierrequintindekercadio/Desktop/MOSEF/S1/projet_time_series/Store_Sales_Time_Series_Forecasting'

In [5]:
sales_df = pd.read_csv('data/sales_df.csv')
sales_df['date'] = pd.to_datetime(sales_df['date'])

In [6]:
sales_df.head(5)

,id,date,store_nbr,family,sales,onpromotion,city,state,store_type,cluster,...,sales_lag_364,rolling_mean_7,rolling_mean_28,rolling_mean_56,rolling_std_7,store_family_velocity,target_enc_store_family,oil_trend_30,promo_ratio_vs_avg,promo_during_payday
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0,Quito,Pichincha,D,13,...,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.00,0.0,0
1,1782,2013-01-02,1,AUTOMOTIVE,2.0,0,Quito,Pichincha,D,13,...,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,0.00,0.0,0
2,3564,2013-01-03,1,AUTOMOTIVE,3.0,0,Quito,Pichincha,D,13,...,NaN,NaN,NaN,NaN,NaN,1.000000,1.000000,-0.17,0.0,0
3,5346,2013-01-04,1,AUTOMOTIVE,3.0,0,Quito,Pichincha,D,13,...,NaN,NaN,NaN,NaN,NaN,1.666667,1.666667,-0.02,0.0,0
4,7128,2013-01-05,1,AUTOMOTIVE,5.0,0,Quito,Pichincha,D,13,...,NaN,NaN,NaN,NaN,NaN,2.000000,2.000000,-0.02,0.0,0


## 1 - BENCHMARK 1 : Naïf Saisonnier 

### On prédit que les ventes seront identiques à celles du même jour la semaine dernière
#### 1.1 Prédiction pour 7 jours

In [ ]:
val_start_date = sales_df['date'].max() - pd.Timedelta(days=7)
train = sales_df[sales_df['date'] < val_start_date]
test = sales_df[sales_df['date'] >= val_start_date]

def rmsle(y_true, y_pred):
    return np.sqrt(mean_squared_log_error(y_true, np.maximum(0, y_pred)))

In [27]:
test['pred_naive_7'] = test['sales_lag_7']
score_7 = rmsle(test['sales'], test['pred_naive_7'].fillna(0))

/var/folders/p0/3rkkfb2j3137v01dmkrd3tx40000gn/T/ipykernel_65687/2453435315.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test['pred_naive_7'] = test['sales_lag_7']


In [20]:
print(f"RMSLE Naïf (J-7): {score_7:.4f}")

RMSLE Naïf (J-7): 0.5694


#### 1.2 Prédiction pour 56 jours ( 8 semaines )

In [30]:
val_start_long = sales_df['date'].max() - pd.Timedelta(days=56)
test_56 = sales_df[sales_df['date'] >= val_start_long].copy()

In [31]:
test_56['pred_naive_56'] = test_56['sales_lag_56']
score_56 = rmsle(test_56['sales'], test_56['pred_naive_56'].fillna(0))

print(f"RMSLE Naïf (J-56) : {score_56:.4f}")

RMSLE Naïf (J-56) : 0.6060


# 2 - Benchmark 2 

#### 2.1 Prédidtion pour 7 jours 

In [ ]:
test['pred_rolling_7'] = test['rolling_mean_7']
score_rolling_7 = rmsle(test['sales'], test['pred_rolling_7'].fillna(0))

/var/folders/p0/3rkkfb2j3137v01dmkrd3tx40000gn/T/ipykernel_65687/4024527553.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test['pred_rolling_7'] = test['rolling_mean_7']


In [ ]:
print(f"RMSLE Moyenne Mobile (7j): {score_rolling_7:.4f}")

RMSLE Moyenne Mobile (7j): 0.4489


#### 2.2 Prédiction pour 56 jours ( 8 semaines )

In [43]:
test_56['pred_ma_56'] = test_56['rolling_mean_56'] 
score_rolling_56 = rmsle(test_56['sales'], test_56['pred_ma_56'].fillna(0))

In [44]:
print(f"RMSLE Moyenne Mobile (56j): {score_rolling_56:.4f}")

RMSLE Moyenne Mobile (56j): 0.4750


# 3 - Résumé

RMSLE Naïf (J-7): **0.5694**

RMSLE Naïf (J-56) : **0.6060**

RMSLE Moyenne Mobile (7j): **0.4489**

RMSLE Moyenne Mobile (56j): **0.4750**